# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# # Adjust this path if your repo is stored elsewhere in Drive.
# PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"

In [ ]:
# Install Python dependencies (run once per session)
%pip install -r requirements.txt -q
!python3 -m spacy download en

---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [ ]:
import sys, os

# if PROJECT_ROOT not in sys.path:
#     sys.path.insert(0, PROJECT_ROOT)

# os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [ ]:
from Tools.download import download_mini

download_mini(data_dir="_data")

---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [ ]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [ ]:
from TrainTools.train import train

results = train(
    # -- data paths (must match preprocess outputs) --
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # -- time-budgeted training loop (target: ~30 min) --
    num_steps             = 3000,
    checkpoint            = 300,
    val_num_batches       = 0,
    test_num_batches      = 80,
    accumulate_grad_steps = 4,
    batch_size            = 8,
    seed                  = 42,
    early_stop            = 8,

    # -- faster-converging recipe for early F1 lift --
    optimizer_name = "adam",
    scheduler_name = "cosine",
    loss_name      = "qa_nll",
    learning_rate  = 1.5e-3,
    beta1          = 0.9,
    beta2          = 0.999,
    eps            = 1e-8,
    weight_decay   = 3e-7,
    warmup_steps   = 250,
    grad_clip      = 1.0,
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

100%|██████████| 300/300 [26:52<00:00,  5.37s/it]


STEP      300  loss 8.048812

VALID(train) skipped (val_num_batches <= 0)



100%|██████████| 80/80 [00:27<00:00,  2.95it/s]


TEST        loss 3.900956  F1 9.515422  EM 4.062500

Learning rate: [0.0014989721510659303, 0.0014989721510659303]
Checkpoint updated at step 300 (F1=9.5154, EM=4.0625)


100%|██████████| 300/300 [27:23<00:00,  5.48s/it]


...

100%|██████████| 300/300 [26:48<00:00,  5.36s/it]


STEP      900  loss 3.442487

VALID(train) skipped (val_num_batches <= 0)



100%|██████████| 80/80 [00:25<00:00,  3.10it/s]


TEST        loss 3.166961  F1 25.258031  EM 17.343750

Learning rate: [0.0013328594710927282, 0.0013328594710927282]
Checkpoint updated at step 900 (F1=25.2580, EM=17.3438)


 57%|█████▋    | 172/300 [15:33<11:47,  5.53s/it]

---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")